In [ ]:
%load_ext autoreload
%autoreload 2

### MS in-distribution updating with 10% additional data.

 - Base models generated with `train_mhcprime_with_10pct_splits.py`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name != "CLSBN26":
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find CLSBN26 project root.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "Data"
SRC_DIR = PROJECT_ROOT / "src"
MODEL_CHECKPOINTS_DIR = PROJECT_ROOT / "Model_Checkpoints"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("PROJECT_ROOT in sys.path:", str(PROJECT_ROOT) in sys.path)

In [ ]:
# imports
from Data.mhc_pseudo_seqs import mhc_seq_dict
from src.general_utils import *
from src.get_feature_data import *
from src.Model.mhcprime import *
from src.model_utils import *
from src.general_utils import *
from src.get_feature_data import *
from src.pepcl import *
from src.mhcprime_inference import *
from src.log_odds import *
from src.plotting import *

# misc
import warnings
warnings.filterwarnings("ignore")
device="cuda"

In [ ]:
# change gpu
GPU_ID = 0
# GPU_ID = 1
# GPU_ID = 2
# GPU_ID = 3

torch.cuda.set_device(GPU_ID)
print(f"Using GPU: {torch.cuda.get_device_name(GPU_ID)} (ID: {torch.cuda.current_device()})")

Using GPU: NVIDIA H100 80GB HBM3 (ID: 0)


### Get datasets

#### Format pseudosequences and get feature data

In [ ]:
# format pseudosequences
mhc_seq_dict={k.removeprefix("HLA-").replace("*","").replace(":",""):v for k,v in mhc_seq_dict.items()}

In [ ]:
# get feature table and processed features
aa_features_table_ext, descriptor_groups=load_feature_data()
processed_feature_table, corr_info = process_aa_feature_table(aa_features_table_ext)
feature_names = list(processed_feature_table.columns)  # pass to FeatureProjectionEncoder

#### MS train and test sets

In [ ]:
# ms train and test sets
datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
ms_train_data=pd.read_csv(f"{datasets_dir}/ms_train_data.csv", index_col=0)
ms_test_data=pd.read_csv(f"{datasets_dir}/ms_test_data.csv", index_col=0)

In [ ]:
# seq allele splits
ms_train_data_seq_allele_dict_10_pct_splits=load_dict_pickle(f"{datasets_dir}/ms_train_data_seq_allele_dict_10_pct_splits.pkl")

In [ ]:
# process
ms_train_data=full_process_data(ms_train_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_train_data=create_seq_allele_column(ms_train_data)

# remove padding from seq allele
ms_train_data.seq_allele=ms_train_data.seq_allele.str.replace("$", "")

In [ ]:
# process if it hasnt been done already
ms_test_data=full_process_data(ms_test_data, mhc_seq_dict, remove_flank=True, add_pseudo_label=False)
ms_test_data=create_seq_allele_column(ms_test_data)

In [ ]:
# get human proteome background
canonical_human_proteins=pd.read_csv(f"{DATA_DIR}/canonical_human_proteins.csv")

### Load MHCPrime and split models

In [ ]:
# setup tokenizer
tokenizer = AminoAcidTokenizer()

In [ ]:
# set base model params
base_model_params={
        'vocab_size': tokenizer.vocab_size,
        'embed_dim': 480,
        'fc_hidden_dims': [240, 120], 
        'fc_dropout': 0.1,
        'n_heads': 16,
        'max_peptide_len': 34,
        'max_mhc_len': 34,
        "n_peptide_layers": 2,
        "n_mhc_layers": 1,
        'n_concat_layers': 3,
        'add_pos_enc': True,
        'use_segment_embedder': True,
        'dim_feedforward_multiplier': 1,
        'debugging': False,
        'fc_output_dim': 1,
        'pooling': 'cls', # cls, mean, min, max, attn, for sequence ffnn head
        "ffnn_use_layernorm": False, 
        'use_feature_table': True,
        "feature_table": processed_feature_table,
        "sequence_embedder_feature_names": feature_names,
        'sequence_embedder_feature_mode': "concat",  # replace or concat
        'sequence_embedder_tokenizer': tokenizer,
        "peptide_start_pos": 1, 
        "mhc_start_pos": 35, 
        "pos_enc_type": "sincos", 
        "seed": 42,

        # peptide conformer
        "add_peptide_conformer": True, 
        "peptide_conformer_num_layers": 2,
        "peptide_conformer_n_heads": 16,
        "peptide_conformer_conv_kernel": 5,
        "peptide_conformer_dropout": 0.0,

        # mhc conformer
        "add_mhc_conformer": False, 
        "mhc_conformer_num_layers": 2,
        "mhc_conformer_n_heads": 16,
        "mhc_conformer_conv_kernel": 5,
        "mhc_conformer_dropout": 0.0,

        # fusion conformer parameters
        "add_fusion_conformer": True, 
        "fusion_conformer_num_layers": 1,
        "fusion_conformer_n_heads": 16,
        "fusion_conformer_conv_kernel": 7,
        "fusion_conformer_dropout": 0.0,
        "fusion_conformer_mask_bos": True,
        "ff_pre_post_swiglu": False,

        # legacy init
        "legacy_init_compat": True,
}

In [ ]:
%%time
# load base model (optionally, can load model_params directly)
# run_name = "MHCPrime_Base"
# run_dir = MODEL_CHECKPOINTS_DIR / run_name
# model_dir = run_dir / "Checkpoints"

# mhcprime=init_model(base_model_params, print_check=False, print_params=True)
# mhcprime.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
# mhcprime.eval()
# print("Model loaded from checkpoint...")
# clear_all_gpu_memory()

base_model_dir = MODEL_CHECKPOINTS_DIR / "MHCPrime_Base" / "Checkpoints"
ckpt_path = base_model_dir / "model_final.pt"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

clear_all_gpu_memory()

mhcprime = init_model(
    base_model_params,
    print_check=False,
    print_params=True,
)

state_dict = torch.load(
    ckpt_path,
    map_location="cpu",
)

mhcprime.load_state_dict(state_dict, strict=True)
mhcprime.to(device)
mhcprime.eval()

print("Model loaded from checkpoint...")
print(f"Checkpoint: {ckpt_path}")
print(f"Model device: {next(mhcprime.parameters()).device}")

[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
Model loaded from checkpoint...
All GPU memory has been cleared.
Current GPU memory allocated: 0.10 GB
Current GPU memory reserved: 0.12 GB
CPU times: user 1.12 s, sys: 472 ms, total: 1.6 s
Wall time: 1.61 s


In [ ]:
# add split models
split_percentages_dict={
    10: [0],
    20: [0, 1],
    30: [0, 1, 2],
    40: [0, 1, 2, 3],
    50: [0, 1, 2, 3, 4],
    60: [0, 1, 2, 3, 4, 5],
    70: [0, 1, 2, 3, 4, 5, 6],
    80: [0, 1, 2, 3, 4, 5, 6, 7],
    90: [0, 1, 2, 3, 4, 5, 6, 7, 8],
}

models={"mhcprime": mhcprime}
for pt_pct in split_percentages_dict.keys():
    model_dir=f"{MODEL_CHECKPOINTS_DIR}/MHCPrime_Base_{str(pt_pct)}pct/Checkpoints"
    model_name=f"mhcprime_{str(pt_pct)}pct"
    print(model_name)
    model=init_model(base_model_params, print_check=False, print_params=True)
    model.load_state_dict(torch.load(f"{model_dir}/model_final.pt"))
    models[model_name]=model

mhcprime_10pct
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_20pct
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_30pct
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_40pct
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_50pct
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loa

### PepCL updating

#### Run PepCL

 - Updating done in `update_mhcprime_with_10pct_more_data.py`.

#### Inference

In [ ]:
# run all base models
base_and_ft_models=models.copy()

In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:21, 39507.52it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:17<00:00,  7.63it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:13<00:00,  7.84it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:15<00:00,  7.75it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:14<00:00,  7.76it/s]


CPU times: user 8h 16min 58s, sys: 2min 2s, total: 8h 19min 1s
Wall time: 24min 10s


In [ ]:
%%time
"""
Load models
"""

# load_models=False
load_models=True

steps = 1000
in_distribution_updating_dir = MODEL_CHECKPOINTS_DIR / f"MS_10pct_more_updating_models_{steps}steps"

if load_models:
    base_and_ft_models={}
    for model_name in os.listdir(in_distribution_updating_dir):
        print(model_name)
        if model_name.endswith(".pt"):
            model_name_cleaned=model_name.removesuffix(".pt")
            print(model_name_cleaned)
            model=init_model(base_model_params, print_check=False, print_params=True)
            model.load_state_dict(torch.load(f"{in_distribution_updating_dir}/{model_name}"))
            base_and_ft_models[model_name_cleaned]=model

print(len(base_and_ft_models))

mhcprime_10pct_updated_on_10pct_ft.pt
mhcprime_10pct_updated_on_10pct_ft
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_30pct_updated_on_10pct_pepcl.pt
mhcprime_30pct_updated_on_10pct_pepcl
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_50pct_updated_on_10pct_ft.pt
mhcprime_50pct_updated_on_10pct_ft
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of trainable parameters: 25751991
Model loaded successfully...
mhcprime_50pct_updated_on_10pct_pepcl.pt
mhcprime_50pct_updated_on_10pct_pepcl
[FeatureProjectionEncoder] Warning: no features found for tokens: ['$', '*', '<BOS>']
Number of trainable modules: 202
Number of 

In [ ]:
%%time
# ms_test_data
build_caches=True
# build_caches=False

if build_caches:
    data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"

    build_token_cache(ms_test_data, data_to_run_path, tokenizer, use_domain_id=False)

data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
for model_name, model in base_and_ft_models.items():
    model.eval()
    with torch.no_grad():
        output_scores=run_mhcprime_fast(model, data_to_run_path, batch_size=3072, num_workers=16, compile_model=False, use_domain_id=False)

    ms_test_data[model_name]=output_scores.detach().cpu().tolist()

3214300it [01:30, 35493.52it/s]


wrote cached tensors → /home/pchati/PepCL_Manuscript/Data/dataset_cache.pt  (874.3 MB)
Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:11<00:00,  7.95it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:16<00:00,  7.67it/s]


...

Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:11<00:00,  7.97it/s]


Getting data
Loading data
Setting up model
Running inference


100%|██████████| 1047/1047 [02:11<00:00,  7.99it/s]


CPU times: user 13h 16min 48s, sys: 3min 42s, total: 13h 20min 31s
Wall time: 41min 38s


In [ ]:
# delete dataset cache
data_to_run_path=f"{DATA_DIR}/dataset_cache.pt"
os.remove(data_to_run_path)

In [ ]:
%%time
"""
Save test set
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

CPU times: user 50.3 s, sys: 1.87 s, total: 52.1 s
Wall time: 52.1 s


### Generate log odds

In [ ]:
%%time
# generate background
chp_filtered_negative_peptides=generate_kmers_from_proteome(canonical_human_proteins, kmer_lengths=[8,9,10,11,12,13,14])
random_negative_peptides=generate_random_peptides(n_peptides=10, lengths=[8,9,10,11,12,13,14])

chp_filtered_negative_peptides_no_dup={}
for l,p in tqdm(chp_filtered_negative_peptides.items()):
    chp_filtered_negative_peptides_no_dup[l]=list(set(p))
    print("Length", l)
    print(len(chp_filtered_negative_peptides[l]))
    print(len(chp_filtered_negative_peptides_no_dup[l]))

 14%|█▍        | 1/7 [00:02<00:15,  2.61s/it]

Length 8
11250173
10300888
Length 9
11229814
10405224


 43%|████▎     | 3/7 [00:07<00:10,  2.64s/it]

Length 10
11209455
10466109
Length 11
11189096
10509168


 86%|████████▌ | 6/7 [00:15<00:02,  2.63s/it]

Length 12
11168738
10540392
Length 13
11148381
10563518


100%|██████████| 7/7 [00:18<00:00,  2.65s/it]

Length 14
11128026
10580237
CPU times: user 1min 10s, sys: 5.05 s, total: 1min 15s
Wall time: 1min 15s


In [ ]:
# create proper columns
ms_train_data["peptide"]=ms_train_data.seq.str.replace("$", "")
ms_train_data["seq_len"]=ms_train_data.peptide.apply(len)
ms_train_data=create_seq_allele_column(ms_train_data)

ms_test_data["peptide"]=ms_test_data.seq.str.replace("$", "")
ms_test_data["seq_len"]=ms_test_data.peptide.apply(len)
ms_test_data=create_seq_allele_column(ms_test_data)

In [ ]:
%%time
# compute ms log odds

ms_train_data.seq_allele=ms_train_data.seq_allele.str.replace("$" ,"")
for pt_pct, split_list in split_percentages_dict.items():
    print(pt_pct, split_list)
    seq_allele_list=merge_splits(ms_train_data_seq_allele_dict_10_pct_splits, split_list)
    print(len(seq_allele_list))

    # filter to seq allele target
    train_ft_data=ms_train_data.copy()
    train_ft_data=train_ft_data[train_ft_data.seq_allele.isin(seq_allele_list)]
    print(train_ft_data.query("label == 1").shape)

    # get ref set
    ref_df=train_ft_data.copy()

    # ms test data
    ms_test_data=compute_multi_allele_length_pwm_logodds_gpu(
        background_peptides_by_length=chp_filtered_negative_peptides_no_dup,
        ref_df=ref_df.query("label == 1"), # pos ms only
        df=ms_test_data,
        allele_col = "allele",
        seq_col = "peptide",
        out_col = f"ms_log_odds_{str(pt_pct)}",
        peptide_lengths=None,
        aa_alphabet = AA_ALPHABET_20,
        pseudocount_bg = 1.0,
        pseudocount_ref = 50.0,
        device = "cuda",
    )


10 [0]
3861640
(75769, 15)


Computing PWM log-odds (allelexlen): 100%|██████████| 994/994 [00:07<00:00, 130.00it/s]


20 [0, 1]
7656573
(150173, 15)


Computing PWM log-odds (allelexlen): 100%|██████████| 994/994 [00:07<00:00, 128.99it/s]


30 [0, 1, 2]
11448386
(224516, 15)


...

80 [0, 1, 2, 3, 4, 5, 6, 7]
30407998
(596244, 15)


Computing PWM log-odds (allelexlen): 100%|██████████| 994/994 [00:08<00:00, 121.89it/s]


90 [0, 1, 2, 3, 4, 5, 6, 7, 8]
34199783
(670587, 15)


Computing PWM log-odds (allelexlen): 100%|██████████| 994/994 [00:08<00:00, 116.32it/s]


CPU times: user 11min 19s, sys: 1min 47s, total: 13min 7s
Wall time: 13min 5s


In [ ]:
%%time
"""
Save test set
"""
save_dfs=False
# save_dfs=True

if save_dfs:
    # ms test set
    datasets_dir = DATA_DIR / "Training_Datasets" / "Processed" / "MS_Train_Test_Sets"
    ms_test_data.to_csv(f"{datasets_dir}/ms_test_data.csv")

CPU times: user 1min 5s, sys: 2.24 s, total: 1min 8s
Wall time: 1min 7s
